# AI-Powered Supply Chain Risk Intelligence Platform

### Notebook 02 - Silver Layer Data Cleaning & Transformation

---

# Overview

The Silver layer transforms raw data from the Bronze layer into a **clean, structured dataset suitable for analytics and machine learning**.

While the Bronze layer preserves the original dataset, the Silver layer focuses on:

* Data validation
* Schema standardization
* Handling inconsistent values
* Creating core operational metrics

These transformations ensure that downstream analytics and machine learning models operate on **reliable and consistent data**.

---

# Objectives of the Silver Layer

The key goals of this notebook are:

• Clean and standardize the raw shipment dataset
• Convert date fields into proper timestamp formats
• Handle malformed or inconsistent values
• Calculate operational logistics metrics
• Prepare the dataset for feature engineering in the Gold layer

The Silver dataset acts as the **foundation for analytics and machine learning pipelines**.

---

# Input Dataset

This notebook reads the Bronze table created in the previous step:

```
supply_chain.default.bronze_shipments
```

This table contains the raw shipment records ingested from the source dataset.

---

# Output Dataset

At the end of this notebook, the cleaned dataset will be stored as:

```
supply_chain.default.silver_shipments
```

This table will be used for **feature engineering and risk analytics in the Gold layer**.


# Loading Bronze Dataset

The first step is to load the raw dataset stored in the Bronze layer.

The Bronze dataset represents the **source-of-truth version of the shipment data**.
All transformations applied in this notebook will operate on this dataset while preserving the original raw data.

By separating Bronze and Silver layers, we ensure that:

• Raw data remains immutable

• Data cleaning logic is transparent

• Transformations are reproducible


In [0]:
df = spark.read.table("supply_chain.default.bronze_shipments")

# Data Cleaning and Standardization

Real-world operational datasets often contain inconsistent formats, missing values, and malformed records.

This step performs several key transformations to improve data quality:

### Date Standardization

Shipment datasets often store dates as text values.
These columns are converted into proper date formats to enable accurate time-based calculations.

### Handling Invalid Values

Certain columns may contain unexpected values (for example text in numeric fields).
These values are safely handled to ensure the dataset remains usable for analytics.

### Schema Standardization

Column names are standardized to ensure compatibility with SQL queries and machine learning workflows.

These steps ensure that the dataset becomes **structured, consistent, and analytics-ready**.


In [0]:
from pyspark.sql.functions import expr

df = df.withColumn(
    "scheduled_delivery_date",
    expr("try_to_timestamp(scheduled_delivery_date, 'd-MMM-yy')")
)

df = df.withColumn(
    "delivered_to_client_date",
    expr("try_to_timestamp(delivered_to_client_date, 'd-MMM-yy')")
)

df = df.withColumn(
    "po_sent_to_vendor_date",
    expr("try_to_timestamp(po_sent_to_vendor_date, 'd-MMM-yy')")
)

df = df.withColumn(
    "pq_first_sent_to_client_date",
    expr("try_to_timestamp(pq_first_sent_to_client_date, 'd-MMM-yy')")
)

df = df.withColumn(
    "delivery_recorded_date",
    expr("try_to_timestamp(delivery_recorded_date, 'd-MMM-yy')")
)

In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("scheduled_delivery_date", to_date("scheduled_delivery_date"))
df = df.withColumn("delivered_to_client_date", to_date("delivered_to_client_date"))
df = df.withColumn("po_sent_to_vendor_date", to_date("po_sent_to_vendor_date"))
df = df.withColumn("pq_first_sent_to_client_date", to_date("pq_first_sent_to_client_date"))
df = df.withColumn("delivery_recorded_date", to_date("delivery_recorded_date"))

# Shipment Delay Calculation

Delivery performance is one of the most critical metrics in supply chain operations.

To evaluate shipment reliability, we compute the **delivery delay metric**:

```
delivery_delay_days = delivered_to_client_date − scheduled_delivery_date
```

This metric helps identify whether shipments were delivered:

• On time

• Earlier than expected

• Delayed

This delay metric will later serve as the **target variable for machine learning models** and will also drive logistics performance analytics in the Gold layer.


In [0]:
from pyspark.sql.functions import datediff, col

df = df.withColumn(
    "delivery_delay_days",
    datediff(col("delivered_to_client_date"), col("scheduled_delivery_date"))
)

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_delayed",
    when(col("delivery_delay_days") > 0, 1).otherwise(0)
)

# Persisting the Silver Dataset

After cleaning and transformation, the dataset is stored as a **Delta table in the Silver layer**.

This dataset now contains:

• Standardized schema

• Cleaned operational data

• Delivery delay metrics

The Silver table will serve as the **input dataset for advanced feature engineering in the Gold layer**, where business intelligence features and machine learning inputs will be created.


In [0]:
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("supply_chain.default.silver_shipments")